# Langchain
## this is a notebook to learn and explore about langchain frameworks

In [1]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.3 MB/s eta 0:00:00


In [2]:
from langchain_groq import ChatGroq
import os

In [3]:
os.environ["GROQ_API_KEY"] = "xxx"

# list of models available in groq

In [4]:
from groq import Groq
import os

# Initialize the client (it will use the API key from your environment)
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Fetch and print the list of models
models = client.models.list()
for model in models.data:
    print(model.id)

openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
groq/compound
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
meta-llama/llama-4-scout-17b-16e-instruct
whisper-large-v3-turbo
groq/compound-mini
llama-3.3-70b-versatile
canopylabs/orpheus-v1-english
allam-2-7b
canopylabs/orpheus-arabic-saudi
qwen/qwen3-32b
llama-3.1-8b-instant


# model configuration
here the llm is the model and using ChatGroq i have modified the parameters.

In [5]:
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

In [6]:
llm.invoke("HI")

AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 36, 'total_tokens': 59, 'completion_time': 0.045655974, 'completion_tokens_details': None, 'prompt_time': 0.001716527, 'prompt_tokens_details': None, 'queue_time': 0.165650961, 'total_time': 0.047372501}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb53f-36c3-7e92-b3ec-6182bf6c1316-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59})

we have given an example conversation and asking the model the continue the convo. using messages - AI, System and human we can specify the covnersation.

SystemMessage - description we give to the model

AIMessage - past convo made by the model (we are describing

HumanMessage - our prompt

In [7]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

llm.invoke(messages)

AIMessage(content='2 + 2 = 4. Is there anything else I can help you with?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 75, 'total_tokens': 94, 'completion_time': 0.030884127, 'completion_tokens_details': None, 'prompt_time': 0.003454603, 'prompt_tokens_details': None, 'queue_time': 0.318569979, 'total_time': 0.03433873}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb53f-3849-7470-ad9e-6f395d97117d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 19, 'total_tokens': 94})

 a small change ending with a ai message so that the llm model ll continue the conversation from where it left.

In [8]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a haiku about spring"),
    AIMessage("Cherry blossoms bloom...")
]
response = llm.invoke(messages)

In [9]:
response

AIMessage(content=" \nDancing petals softly fall \nSpring's gentle delight", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 52, 'total_tokens': 64, 'completion_time': 0.046077316, 'completion_tokens_details': None, 'prompt_time': 0.002396744, 'prompt_tokens_details': None, 'queue_time': 0.161201026, 'total_time': 0.04847406}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb53f-3a14-7d82-9ab4-76519f853653-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 12, 'total_tokens': 64})

In [10]:
response = llm.invoke("Explain AI")
print(type(response))  # <class 'langchain.messages.AIMessage'>

<class 'langchain_core.messages.ai.AIMessage'>


## Tool calling

we use model_with_tools to bind llm + with our tools created.

In [11]:
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    ...

model_with_tools = llm.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather in Paris?")

In [12]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

Tool: get_weather
Args: {'location': 'Paris'}
ID: n28s68g36


In [13]:
response.usage_metadata

{'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233}

In [14]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = llm.invoke(messages)  # Model processes the result

In [15]:
response

AIMessage(content='The current weather in San Francisco is sunny with a temperature of 72°F.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 74, 'total_tokens': 91, 'completion_time': 0.052448868, 'completion_tokens_details': None, 'prompt_time': 0.006482165, 'prompt_tokens_details': None, 'queue_time': 0.049881151, 'total_time': 0.058931033}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb53f-457c-7200-8562-57e11816ea0f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 17, 'total_tokens': 91})

## inspecting response

In [16]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [17]:
response = llm.invoke("What is LangChain?")

print(response.content)

LangChain is an open-source Python library for building large language models (LLMs) and multimodal models. It was created by Joshua Brower and is maintained by the LangChain community. LangChain provides a set of tools and APIs that make it easier to work with LLMs and multimodal models, allowing developers to build more complex and powerful applications.

Some of the key features of LangChain include:

1. **LLM Integration**: LangChain provides a simple and consistent API for integrating with popular LLMs such as LLaMA, BERT, and RoBERTa.
2. **Multimodal Support**: LangChain allows developers to work with multimodal models that can process both text and images, audio, or other forms of data.
3. **Chainable Models**: LangChain's core concept is the "chain," which allows developers to create complex models by chaining together multiple LLMs or multimodal models.
4. **Data Loading and Preprocessing**: LangChain provides tools for loading and preprocessing data, making it easier to work 

In [18]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [19]:
print(response)

content='LangChain is an open-source Python library for building large language models (LLMs) and multimodal models. It was created by Joshua Brower and is maintained by the LangChain community. LangChain provides a set of tools and APIs that make it easier to work with LLMs and multimodal models, allowing developers to build more complex and powerful applications.\n\nSome of the key features of LangChain include:\n\n1. **LLM Integration**: LangChain provides a simple and consistent API for integrating with popular LLMs such as LLaMA, BERT, and RoBERTa.\n2. **Multimodal Support**: LangChain allows developers to work with multimodal models that can process both text and images, audio, or other forms of data.\n3. **Chainable Models**: LangChain\'s core concept is the "chain," which allows developers to create complex models by chaining together multiple LLMs or multimodal models.\n4. **Data Loading and Preprocessing**: LangChain provides tools for loading and preprocessing data, making i

#Prompt template

In [20]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms."
)

formatted_prompt = prompt.invoke(
    {"topic": "LangChain"}
)

print(formatted_prompt)

messages=[HumanMessage(content='Explain LangChain in simple terms.', additional_kwargs={}, response_metadata={})]


In [21]:
chain = prompt | llm

response = chain.invoke(
    {"topic": "LangGraph"}
)

print(response.content)

LangGraph is a tool used in natural language processing (NLP) to analyze and visualize the structure of languages. It's a graph-based approach that represents words and their relationships in a language as nodes and edges in a graph.

Here's a simple breakdown:

1. **Nodes**: Each word in a language is represented as a node in the graph. These nodes are connected to other nodes based on their relationships.
2. **Edges**: The edges between nodes represent the relationships between words, such as:
	* Synonyms (words with similar meanings)
	* Antonyms (words with opposite meanings)
	* Hyponyms (words with more specific meanings)
	* Hypernyms (words with more general meanings)
	* Collocations (words that often appear together)
3. **Graph structure**: The graph structure is used to visualize the relationships between words and their meanings. This can help identify patterns, clusters, and hierarchies in the language.

LangGraph is useful for various NLP tasks, such as:

1. **Text analysis**

In [22]:
chain = prompt | llm

response = chain.invoke(
    {"topic": "RAG"}
)

print(type(chain))
print(response.content)

<class 'langchain_core.runnables.base.RunnableSequence'>
RAG is a simple way to categorize or track the status of something, often used in project management, quality control, or customer service. It stands for:

- **R**ed: This indicates a problem or issue that needs immediate attention. It's like a warning sign that says "something's wrong, fix it now."
- **A**mber: This means there's a potential issue or a warning sign that needs to be monitored. It's like a caution sign that says "be careful, something might go wrong."
- **G**reen: This indicates that everything is okay, and there are no issues or problems. It's like a green light that says "all clear, everything's fine."

Using RAG can help teams or individuals quickly identify and prioritize tasks, issues, or problems, and make decisions about what needs to be done next.


# Chains & Output parser

In [23]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [24]:
chain = prompt | llm | parser

In [25]:
response = chain.invoke(
    {"topic": "LangChain"}
)

print(type(response))
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
**What is LangChain?**

LangChain is an open-source Python library that helps developers build conversational AI applications. It's designed to make it easier to create chatbots, virtual assistants, and other language-based models.

**Key Features:**

1. **Modular Architecture**: LangChain allows you to break down complex conversational flows into smaller, reusable components. This makes it easier to manage and maintain your code.
2. **Chainable Functions**: LangChain's core concept is the "chain," which is a sequence of functions that can be executed in a specific order. This enables you to create complex conversational flows by chaining together simple functions.
3. **Integration with Popular Libraries**: LangChain supports integration with popular libraries like Hugging Face Transformers, spaCy, and more. This makes it easy to leverage pre-trained models and tools in your conversational AI applications.
4. **Easy Debugging**: LangC

In [26]:
response = chain.invoke(
    {"topic": "RAG"}
)

print(repr(response))


'RAG is a simple way to categorize or track the status of something, often used in project management, quality control, or customer service. It stands for:\n\n- **R**ed: This indicates a problem or issue that needs immediate attention. It\'s like a warning sign that says "something\'s wrong, fix it now."\n- **A**mber: This means there\'s a potential issue or a warning sign that needs to be monitored. It\'s like a caution sign that says "be careful, something might go wrong."\n- **G**reen: This indicates that everything is okay, and there are no issues or problems. It\'s like a green light that says "all clear, everything\'s fine."\n\nUsing RAG can help teams or individuals quickly identify and prioritize tasks, issues, or problems, and make decisions about what needs to be done next.'


## structured output parser

In [27]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="Person name")
    age: int = Field(description="Person age")

In [28]:
structured_llm = llm.with_structured_output(Person)

response = structured_llm.invoke(
    "John is 25 years old."
)

print(response)
print(type(response))

name='John' age=25
<class '__main__.Person'>


In [41]:
response = chain.invoke(
    {"topic": "Retrieval Augmented Generation (RAG)"}
)

print(type(response))
print(repr(response))

<class 'langchain_core.messages.base.TextAccessor'>
'**Retrieval Augmented Generation (RAG)**\n\nRetrieval Augmented Generation (RAG) is a type of artificial intelligence (AI) model that combines the strengths of two different approaches: **Retrieval** and **Generation**.\n\n**Retrieval** involves searching through a large database or knowledge base to find relevant information related to a specific query or prompt. Think of it like searching for a specific book in a massive library.\n\n**Generation**, on the other hand, involves creating new text or content based on a prompt or input. This is like writing a new book based on your imagination and creativity.\n\n**RAG** combines these two approaches by first searching through a large database to find relevant information (Retrieval) and then using that information to generate new text or content (Generation). This process is often referred to as "retrieve and generate".\n\nHere\'s a simple example:\n\n1. You ask a RAG model to write a s

In [44]:
print(isinstance(response, str))

True


# Tools

In [45]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [46]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


just by using tools no llm here

In [47]:
print(multiply.invoke({
    "a": 5,
    "b": 10
}))

50


#Model llm

# New Section

In [48]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

## bind tools

In [49]:
llm_with_tools = llm.bind_tools([multiply])

In [50]:
response = llm_with_tools.invoke(
    "What is 25 multiplied by 4?"
)

print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'xs06ct0qp', 'function': {'arguments': '{"a":25,"b":4}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 221, 'total_tokens': 240, 'completion_time': 0.034487816, 'completion_tokens_details': None, 'prompt_time': 0.069361655, 'prompt_tokens_details': None, 'queue_time': 0.053599098, 'total_time': 0.103849471}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019eb557-074f-7773-ab22-4794cbfcfb45-0' tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 4}, 'id': 'xs06ct0qp', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 221, 'output_tokens': 19, 'total_tokens': 240}


In [51]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 25, 'b': 4},
  'id': 'xs06ct0qp',
  'type': 'tool_call'}]

# Agents

In [55]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=[multiply]
)

/tmp/ipykernel_1290/1372917622.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [56]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is 25 multiplied by 4?"
            }
        ]
    }
)

print(result)

{'messages': [HumanMessage(content='What is 25 multiplied by 4?', additional_kwargs={}, response_metadata={}, id='267bfab4-d02a-449b-abe6-518b44e542d9'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '75b9cj976', 'function': {'arguments': '{"a":25,"b":4}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 221, 'total_tokens': 240, 'completion_time': 0.039130819, 'completion_tokens_details': None, 'prompt_time': 0.706257667, 'prompt_tokens_details': None, 'queue_time': 0.244280381, 'total_time': 0.745388486}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb568-e93d-7ea0-8fcb-83ebfd8b676a-0', tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 4}, 'id': '75b9cj976', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, '

In [57]:
result["messages"]

[HumanMessage(content='What is 25 multiplied by 4?', additional_kwargs={}, response_metadata={}, id='267bfab4-d02a-449b-abe6-518b44e542d9'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '75b9cj976', 'function': {'arguments': '{"a":25,"b":4}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 221, 'total_tokens': 240, 'completion_time': 0.039130819, 'completion_tokens_details': None, 'prompt_time': 0.706257667, 'prompt_tokens_details': None, 'queue_time': 0.244280381, 'total_time': 0.745388486}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb568-e93d-7ea0-8fcb-83ebfd8b676a-0', tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 4}, 'id': '75b9cj976', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_token

# concepts and summary

| Concept   | Purpose               |
| --------- | --------------------- |
| Prompt    | Tell model what to do |
| Chain     | Connect components    |
| Parser    | Structure outputs     |
| Tool      | External capability   |
| Tool Call | LLM decision          |
| Agent     | Execute tool calls    |
| LangGraph | Orchestrate workflow  |


## excercise

In [58]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

added two tools in one agent

In [61]:
agent = create_react_agent(
    model=llm,
    tools=[multiply, add]
)

/tmp/ipykernel_1290/3196038770.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [64]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is 25 multiplied by 4 and then add 50?"
            }
        ]
    }
)

print(result)

{'messages': [HumanMessage(content='What is 25 multiplied by 4 and then add 50?', additional_kwargs={}, response_metadata={}, id='cb3b20c2-f01b-48a7-b535-283992aabf1e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kb5adeccs', 'function': {'arguments': '{"a":25,"b":4}', 'name': 'multiply'}, 'type': 'function'}, {'id': 'wgset04ed', 'function': {'arguments': '{"a":100,"b":50}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 277, 'total_tokens': 313, 'completion_time': 0.057482786, 'completion_tokens_details': None, 'prompt_time': 0.037853937, 'prompt_tokens_details': None, 'queue_time': 0.061687787, 'total_time': 0.095336723}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb573-d3c1-7fb3-a0ca-6cd2df1736eb-0', tool_calls=[{'name': 'multiply', 'args': {'a

In [65]:
result["messages"]

[HumanMessage(content='What is 25 multiplied by 4 and then add 50?', additional_kwargs={}, response_metadata={}, id='cb3b20c2-f01b-48a7-b535-283992aabf1e'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kb5adeccs', 'function': {'arguments': '{"a":25,"b":4}', 'name': 'multiply'}, 'type': 'function'}, {'id': 'wgset04ed', 'function': {'arguments': '{"a":100,"b":50}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 277, 'total_tokens': 313, 'completion_time': 0.057482786, 'completion_tokens_details': None, 'prompt_time': 0.037853937, 'prompt_tokens_details': None, 'queue_time': 0.061687787, 'total_time': 0.095336723}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb573-d3c1-7fb3-a0ca-6cd2df1736eb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 

In [67]:
from langchain_core.messages import HumanMessage

agent.invoke({
    "messages": [
        HumanMessage("What is 25 multiplied by 4 and then add 10?")
    ]
})

{'messages': [HumanMessage(content='What is 25 multiplied by 4 and then add 10?', additional_kwargs={}, response_metadata={}, id='0e46d78f-9f9c-4b93-aae1-1eaf3418872b'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2vgjpwp0c', 'function': {'arguments': '{"a":25,"b":4}', 'name': 'multiply'}, 'type': 'function'}, {'id': '310cc6d6j', 'function': {'arguments': '{"a":100,"b":10}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 277, 'total_tokens': 313, 'completion_time': 0.096404136, 'completion_tokens_details': None, 'prompt_time': 0.022881163, 'prompt_tokens_details': None, 'queue_time': 0.074447561, 'total_time': 0.119285299}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb59c-9b77-7140-a057-3dfe93d29201-0', tool_calls=[{'name': 'multiply', 'args': {

In [74]:
result = agent.invoke({
    "messages": [
        HumanMessage("My name is Dharmendra"),
        HumanMessage("What is my name?")
    ]
})

In [71]:
result

{'messages': [HumanMessage(content='My name is Dharmendra', additional_kwargs={}, response_metadata={}, id='6fd21653-fdc0-4fdc-b024-d88219151ad8'),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='d3d9a374-3f1e-49db-a251-742ae9abccd9'),
  AIMessage(content='Dharmendra', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 279, 'total_tokens': 284, 'completion_time': 0.008585093, 'completion_tokens_details': None, 'prompt_time': 0.018343545, 'prompt_tokens_details': None, 'queue_time': 0.159513155, 'total_time': 0.026928638}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb59d-3e1c-7853-9764-82750574d4a4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 279, 'output_tokens': 5, 'total_tokens': 284})]}

# state vs MEmory

In [76]:
from langgraph.checkpoint.memory import MemorySaver

In [77]:
memory = MemorySaver()

In [78]:
agent = create_react_agent(
    model=llm,
    tools=[],
    checkpointer=memory
)

/tmp/ipykernel_1290/685734041.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [79]:
agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Dharmendra"
            }
        ]
    },
    config={
        "configurable": {
            "thread_id": "user-1"
        }
    }
)

{'messages': [HumanMessage(content='My name is Dharmendra', additional_kwargs={}, response_metadata={}, id='141ab2e5-e8b7-490b-b6c4-28b21132030d'),
  AIMessage(content='Nice to meet you, Dharmendra. Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 41, 'total_tokens': 66, 'completion_time': 0.034364326, 'completion_tokens_details': None, 'prompt_time': 0.072708334, 'prompt_tokens_details': None, 'queue_time': 0.231621929, 'total_time': 0.10707266}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb5a7-f206-7662-8732-fcc8b90ee63a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 25, 'total_tokens': 66})]}

In [81]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my name?"
            }
        ]
    },
    config={
        "configurable": {
            "thread_id": "user-1"
        }
    }
)

print(result["messages"][-1].content)

Your name is Dharmendra.
